# GTA 6 launch night on Kick: reading the curve

Kick.com reports concurrent viewers for *right now* and keeps no history, so the hour-by-hour shape of the GTA 6 launch was never stored by the platform it happened on. This dataset is 22 hourly snapshots taken while it happened.

> **Collected with the [Kick Scraper](https://apify.com/bhole_krishna/kick-scraper) on Apify.**
> Every row here came from that actor, running hourly through the launch. Because Kick keeps no
> history, the hours after this dataset ends can only be obtained by collecting them live - point
> it at any category, put it on a schedule, and you are building the same record for whatever
> launches next: `apify.com/bhole_krishna/kick-scraper`

Four questions this notebook answers:

1. What did the curve look like?
2. When exactly did GTA 6 overtake Just Chatting, Kick's permanent number one?
3. Which language was the launch actually in?
4. Did a handful of huge streamers carry the 1.35M, or the long tail?

The fourth answer is the least obvious, and it reframes the other three.


In [ ]:
import glob
import os

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Found by name rather than by a hard-coded dataset slug, so this notebook keeps
# working if the dataset is renamed or attached alongside others.
def find(filename):
    hits = glob.glob(f"/kaggle/input/**/{filename}", recursive=True) or glob.glob(f"**/{filename}", recursive=True)
    if not hits:
        raise FileNotFoundError(f"{filename} not found - is the dataset attached to this notebook?")
    return hits[0]

hourly = pd.read_csv(find("gta6_viewers_hourly.csv"), parse_dates=["snapshot_utc"])
streams = pd.read_csv(find("gta6_streams_hourly.csv"), parse_dates=["snapshot_utc"])
categories = pd.read_csv(find("kick_categories_hourly.csv"), parse_dates=["snapshot_utc"])

print(f"{len(hourly)} snapshots, {len(streams):,} stream observations, {len(categories)} category rows")
hourly.head()

## 1. The curve

Only the on-the-hour snapshots are plotted, so the spacing is even. The dataset also holds one 07:48 reading from the run that set the hourly schedule up.

In [ ]:
BLUE, ORANGE, GREEN = "#2a78d6", "#eb6834", "#1baf7a"

h = hourly[hourly["snapshot_utc"].dt.minute == 30].sort_values("snapshot_utc")

fig, ax = plt.subplots(figsize=(12, 5.5))
for column, colour, label in [
    ("gta6_viewers", BLUE, "GTA 6"),
    ("just_chatting_viewers", ORANGE, "Just Chatting"),
    ("gta5_viewers", GREEN, "GTA V"),
]:
    ax.plot(h["snapshot_utc"], h[column], color=colour, linewidth=2, label=label)

peak = h.loc[h["gta6_viewers"].idxmax()]
ax.scatter([peak["snapshot_utc"]], [peak["gta6_viewers"]], color=BLUE, zorder=3, s=45)
ax.annotate(
    f"peak {int(peak['gta6_viewers']):,}",
    xy=(peak["snapshot_utc"], peak["gta6_viewers"]),
    xytext=(0, 14), textcoords="offset points", ha="center", fontweight="bold",
)

ax.set_title("GTA 6 launch night on Kick.com", fontsize=15, fontweight="bold", loc="left")
ax.set_ylabel("Concurrent viewers")
ax.set_xlabel("18-19 November 2026 (UTC)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v/1e6:.1f}M" if v >= 1e6 else f"{v/1e3:.0f}k"))
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

first = h.iloc[0]
print(f"start  {int(first['gta6_viewers']):>9,}  rank #{int(first['gta6_rank'])}  at {first['snapshot_utc']:%H:%M} UTC")
print(f"peak   {int(peak['gta6_viewers']):>9,}  rank #{int(peak['gta6_rank'])}  at {peak['snapshot_utc']:%H:%M} UTC")
print(f"growth {peak['gta6_viewers'] / first['gta6_viewers']:.0f}x in "
      f"{(peak['snapshot_utc'] - first['snapshot_utc']).total_seconds() // 3600:.0f} hours")

## 2. The hour it took the top spot

Just Chatting is Kick's standing number one. Rather than eyeball the crossing, find the last snapshot where GTA 6 was behind it and the first where it was ahead.

In [ ]:
h = h.copy()
h["ahead"] = h["gta6_viewers"] > h["just_chatting_viewers"]
crossing = h["ahead"].idxmax()
before, after = h.loc[:crossing].iloc[-2], h.loc[crossing]

for row, word in [(before, "behind"), (after, "ahead ")]:
    print(f"{row['snapshot_utc']:%d %b %H:%M}  GTA 6 {int(row['gta6_viewers']):>9,}  "
          f"Just Chatting {int(row['just_chatting_viewers']):>8,}  ({word}, rank #{int(row['gta6_rank'])})")

gained = int(after["gta6_viewers"] - before["gta6_viewers"])
print(f"\nGTA 6 added {gained:,} concurrent viewers in that one hour.")
print("It held rank #1 for every remaining snapshot:",
      bool((h.loc[crossing:, "gta6_rank"] == 1).all()))

## 3. Which language was the launch in?

Snapshots are hourly, so summing concurrent viewers per language approximates viewer-hours. This is the result most coverage of the launch would not predict.

In [ ]:
by_language = (streams.groupby("language")["viewers"].sum()
               .sort_values(ascending=False))
top = by_language.head(10)

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.barh(top.index[::-1], top.values[::-1], color=BLUE, height=0.7)
ax.set_title("GTA 6 viewer-hours on Kick, by broadcast language", fontsize=14, fontweight="bold", loc="left")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v/1e6:.1f}M"))
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

share = by_language / by_language.sum() * 100
for language, value in top.head(6).items():
    print(f"{language:<22}{int(value):>12,}   {share[language]:4.1f}%")
print(f"\n{len(by_language)} languages in total. English finished "
      f"#{list(by_language.index).index('English') + 1}.")

## 4. Who actually held the 1.35 million?

Up to 899 channels streamed GTA 6 at once. The question is whether the audience sat with a few enormous streamers or spread across the tail.

In [ ]:
peak_hour = streams[streams["snapshot_utc"] == peak["snapshot_utc"]].sort_values("viewers", ascending=False)
total = peak_hour["viewers"].sum()

print(f"{peak['snapshot_utc']:%d %b %H:%M} UTC - {len(peak_hour)} live GTA 6 streams, "
      f"{total:,} viewers between them\n")
for n in (1, 5, 10, 50, 100):
    print(f"top {n:>3} channels hold {peak_hour['viewers'].head(n).sum() / total * 100:5.1f}% of the audience")

print("\nBiggest streams that hour:")
print(peak_hour[["username", "viewers", "language", "stream_title"]]
      .head(10).to_string(index=False, max_colwidth=46))

## Where the data came from, and how to extend it

Every row was collected from Kick.com's own public endpoints with the [**Kick Scraper**](https://apify.com/bhole_krishna/kick-scraper) on Apify, on an hourly schedule. No login and nothing behind a paywall.

The series stops at 19 November 05:30 UTC. Because Kick keeps no history, **the hours after that can only be obtained by collecting them as they happen**. The input behind the streamer-level table was just:

```json
{
  "categories": ["grand-theft-auto-vi"],
  "scrapeLiveStreams": true,
  "maxItems": 1000
}
```

Put that on an hourly schedule and you are building the same record for whatever launches next.

### Two caveats worth respecting

**Snapshots are instants, not averages.** Each row is one reading of a live counter; a peak between two snapshots is not in this data.

**Summed streams land 0-4% under the category total**, because the two figures are read seconds apart and the stream listing does not reach the smallest streams in the long tail. That gap is the honest error bar - it is reported rather than smoothed away.

Full column documentation: [github.com/krishnadubey965/gta6-kick-dataset](https://github.com/krishnadubey965/gta6-kick-dataset)